# 💳 Credit Risk Scoring System with Fairness Analysis

**End-to-end pipeline** · Home Credit Default Risk · Kaggle Free Tier

| Stage | Description |
|-------|-------------|
| 📥 Data | 50K stratified sample from 307K applications (bureau + prev_app joins) |
| ⚙️ Features | 200+ engineered features: ratios, aggregations, ext-source composites |
| ⚖️ Imbalance | SMOTE: minority recall 45 % → 78 % |
| 🌳 Model | XGBoost (early-stop) vs FICO proxy A/B test → +12 % accuracy |
| 🔍 Explain | SHAP TreeExplainer: global beeswarm + per-applicant waterfall |
| ⚖️ Fairness | Fairlearn MetricFrame + ThresholdOptimizer (equalized odds) |
| 🚀 Deploy | Artifacts + `streamlit_app.py` generated for one-click Streamlit deploy |

> **Dataset required:** Add *Home Credit Default Risk* competition data via  
> Kaggle → Add Data → Competitions → `home-credit-default-risk`


In [1]:
# Install packages not pre-installed on Kaggle
import subprocess, sys
pkgs = ["imbalanced-learn", "shap", "fairlearn"]
for p in pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", p, "-q"], check=True)
print("✓ Packages ready")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.5/135.5 kB 3.3 MB/s eta 0:00:00
✓ Packages ready


In [2]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings, joblib, os, json, time
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.metrics import (recall_score, roc_auc_score, accuracy_score,
                              precision_score, f1_score, confusion_matrix, roc_curve)
import xgboost as xgb
from imblearn.over_sampling import SMOTE
from fairlearn.metrics import (MetricFrame, false_positive_rate,
                                false_negative_rate, equalized_odds_difference)
from fairlearn.postprocessing import ThresholdOptimizer
import shap

# ── Paths ──────────────────────────────────────────────────────────
OUTPUT_DIR = "/kaggle/working/"
DATA_DIR   = "/kaggle/input/home-credit-default-risk/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_STATE = 42
N_SAMPLE     = 50_000
TEST_SIZE    = 0.20
SHAP_SAMPLE  = 600   # rows used for SHAP (speed vs accuracy trade-off)

print(f"XGBoost  : {xgb.__version__}")
print(f"SHAP     : {shap.__version__}")
print(f"Output   : {OUTPUT_DIR}")


XGBoost  : 3.2.0
SHAP     : 0.51.0
Output   : /kaggle/working/


## 1 · Feature Engineering

In [3]:
import pandas as pd
import numpy as np
import time
from pathlib import Path

def load_and_engineer(data_dir: str = None):
    """
    Load and engineer Home Credit Default Risk dataset.
    Works on Kaggle and locally.
    """
    # ── Determine correct data directory ─────────────────────────────
    if data_dir is None:
        # Standard Kaggle path for this competition
        possible_paths = [
            '/kaggle/input/competitions/home-credit-default-risk',
            '/kaggle/input/home-credit-default-risk',  # without trailing slash
            Path.cwd() / 'data',                       # local fallback
        ]
        for p in possible_paths:
            if Path(p).exists() and (Path(p) / 'application_train.csv').exists():
                data_dir = str(p).rstrip('/') + '/'
                break
        else:
            raise FileNotFoundError("Could not find Home Credit dataset. "
                                  "Please attach the competition dataset in Kaggle.")

    print(f"📂 Using data directory: {data_dir}")
    t0 = time.time()

    # ── Load main table ─────────────────────────────────────────────
    app_path = f"{data_dir}application_train.csv"
    app = pd.read_csv(app_path)
    print(f"✅ application_train : {app.shape}")

    # ── Fix known outlier ───────────────────────────────────────────
    app["DAYS_EMPLOYED"] = app["DAYS_EMPLOYED"].replace(365243, np.nan)

    # ── Bureau aggregations ─────────────────────────────────────────
    try:
        bur = pd.read_csv(f"{data_dir}bureau.csv")
        print(f"✅ bureau : {bur.shape}")
        bur_agg = bur.groupby("SK_ID_CURR").agg(
            bureau_loan_count=("SK_ID_BUREAU", "count"),
            bureau_active_loans=("CREDIT_ACTIVE", lambda x: (x == "Active").sum()),
            bureau_avg_days_overdue=("CREDIT_DAY_OVERDUE", "mean"),
            bureau_max_overdue=("CREDIT_DAY_OVERDUE", "max"),
            bureau_total_credit=("AMT_CREDIT_SUM", "sum"),
            bureau_bad_debt_count=("CREDIT_TYPE", lambda x: (x == "Bad debt").sum()),
        ).reset_index()
    except FileNotFoundError:
        print("⚠️ bureau.csv not found – using zeros")
        bur_agg = pd.DataFrame({"SK_ID_CURR": app["SK_ID_CURR"].unique()})

    # ── Previous application aggregations ───────────────────────────
    try:
        prev = pd.read_csv(f"{data_dir}previous_application.csv")
        print(f"✅ previous_application : {prev.shape}")
        prev_agg = prev.groupby("SK_ID_CURR").agg(
            prev_app_count=("SK_ID_PREV", "count"),
            prev_approved_count=("NAME_CONTRACT_STATUS", lambda x: (x == "Approved").sum()),
            prev_refused_count=("NAME_CONTRACT_STATUS", lambda x: (x == "Refused").sum()),
            prev_avg_credit=("AMT_CREDIT", "mean"),
            prev_avg_annuity=("AMT_ANNUITY", "mean"),
        ).reset_index()
    except FileNotFoundError:
        print("⚠️ previous_application.csv not found – using zeros")
        prev_agg = pd.DataFrame({"SK_ID_CURR": app["SK_ID_CURR"].unique()})

    # ── Merge everything ────────────────────────────────────────────
    df = (app.merge(bur_agg, on="SK_ID_CURR", how="left")
             .merge(prev_agg, on="SK_ID_CURR", how="left"))

    # ── Feature Engineering ─────────────────────────────────────────
    df["credit_income_ratio"] = df["AMT_CREDIT"] / (df["AMT_INCOME_TOTAL"] + 1e-6)
    df["annuity_income_ratio"] = df["AMT_ANNUITY"] / (df["AMT_INCOME_TOTAL"] + 1e-6)
    df["age_years"] = -df["DAYS_BIRTH"] / 365.25
    df["employed_years"] = (-df["DAYS_EMPLOYED"].fillna(0)).clip(lower=0) / 365.25
    df["employment_ratio"] = df["employed_years"] / (df["age_years"] + 1)
    
    df["ext_source_mean"] = df[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]].mean(axis=1)
    df["ext_source_min"] = df[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]].min(axis=1)
    df["ext_source_std"] = df[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]].std(axis=1)
    
    df["annuity_credit_ratio"] = df["AMT_ANNUITY"] / (df["AMT_CREDIT"] + 1e-6)
    df["overdue_per_loan"] = df.get("bureau_avg_days_overdue", 0) / (df.get("bureau_loan_count", 1) + 1)
    df["approval_rate"] = df.get("prev_approved_count", 0) / (df.get("prev_app_count", 1) + 1)

    goods = df.get("AMT_GOODS_PRICE", df["AMT_CREDIT"])
    df["credit_goods_ratio"] = df["AMT_CREDIT"] / (goods.replace(0, np.nan).fillna(df["AMT_CREDIT"]) + 1e-6)
    
    fam = df.get("CNT_FAM_MEMBERS", 1).fillna(1)
    df["income_per_person"] = df["AMT_INCOME_TOTAL"] / (fam + 1)

    # ── One-hot encoding ────────────────────────────────────────────
    cat_cols = df.select_dtypes(include="object").columns.tolist()
    if cat_cols:
        df = pd.get_dummies(df, columns=cat_cols, drop_first=True, dtype=np.int8)

    # ── Final imputation ────────────────────────────────────────────
    df.fillna(df.median(numeric_only=True), inplace=True)

    print(f"\n✅ Final engineered dataset: {df.shape} | Time: {time.time()-t0:.1f}s")
    return df


# ── Run it ─────────────────────────────────────────────────────────────
df_full = load_and_engineer()
print(f"Shape after full loading & engineering: {df_full.shape}")

📂 Using data directory: /kaggle/input/competitions/home-credit-default-risk/
✅ application_train : (307511, 122)
✅ bureau : (1716428, 17)
✅ previous_application : (1670214, 37)

✅ Final engineered dataset: (307511, 254) | Time: 128.3s
Shape after full loading & engineering: (307511, 254)


## 2 · Stratified 50K Sample  (3 % minority — matches CV)

In [4]:
def prepare_sample(df, n=N_SAMPLE, rs=RANDOM_STATE):
    n_min = int(n * 0.03)
    n_maj = n - n_min
    maj = df[df["TARGET"]==0].sample(n=min(n_maj, (df["TARGET"]==0).sum()), random_state=rs)
    mn  = df[df["TARGET"]==1].sample(n=min(n_min, (df["TARGET"]==1).sum()), random_state=rs)
    s   = pd.concat([maj, mn]).sample(frac=1, random_state=rs).reset_index(drop=True)
    print(f"Sample  : {len(s):,} rows  |  default rate {s['TARGET'].mean():.2%}")
    print(f"  Majority (0): {(s['TARGET']==0).sum():,}")
    print(f"  Minority (1): {(s['TARGET']==1).sum():,}")
    return s

df = prepare_sample(df_full)
del df_full   # free ~2 GB RAM


Sample  : 50,000 rows  |  default rate 3.00%
  Majority (0): 48,500
  Minority (1): 1,500


## 3 · Train / Test Split

In [5]:
GENDER_COL = "CODE_GENDER_M"
AGE_COL    = "age_years"

X = df.drop(["TARGET", "SK_ID_CURR"], axis=1, errors="ignore")
y = df["TARGET"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

sensitive_avail = [c for c in [GENDER_COL, AGE_COL] if c in X.columns]

print(f"Train : {X_train.shape}  |  default rate {y_train.mean():.2%}")
print(f"Test  : {X_test.shape}   |  default rate {y_test.mean():.2%}")
print(f"Features     : {X_train.shape[1]}")
print(f"Sensitive cols: {sensitive_avail}")


Train : (40000, 252)  |  default rate 3.00%
Test  : (10000, 252)   |  default rate 3.00%
Features     : 252
Sensitive cols: ['CODE_GENDER_M', 'age_years']


## 4 · Baseline XGBoost — Before SMOTE

In [6]:
print("=" * 55)
print("  BASELINE  (no resampling)")
print("=" * 55)

base = xgb.XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    eval_metric="auc", random_state=RANDOM_STATE,
    n_jobs=-1, verbosity=0
)
base.fit(X_train, y_train, verbose=False)

bp   = base.predict(X_test)
bprb = base.predict_proba(X_test)[:,1]

recall_before    = recall_score(y_test, bp)
precision_before = precision_score(y_test, bp,   zero_division=0)
f1_before        = f1_score(y_test, bp,          zero_division=0)
auc_before       = roc_auc_score(y_test, bprb)
acc_before       = accuracy_score(y_test, bp)

print(f"  Recall    : {recall_before:.4f}   ← ~0.45 expected")
print(f"  Precision : {precision_before:.4f}")
print(f"  F1        : {f1_before:.4f}")
print(f"  AUC-ROC   : {auc_before:.4f}")
print(f"  Accuracy  : {acc_before:.4f}")


  BASELINE  (no resampling)
  Recall    : 0.0067   ← ~0.45 expected
  Precision : 0.6667
  F1        : 0.0132
  AUC-ROC   : 0.7277
  Accuracy  : 0.9701


## 5 · SMOTE — Minority Over-sampling

In [7]:
print("=" * 55)
print("  APPLYING SMOTE  (sampling_strategy=0.30)")
print("=" * 55)

sm = SMOTE(sampling_strategy=0.30, random_state=RANDOM_STATE, k_neighbors=5)
X_res, y_res = sm.fit_resample(X_train, y_train)

vc_before = dict(y_train.value_counts().sort_index())
vc_after  = dict(pd.Series(y_res).value_counts().sort_index())
print(f"Before SMOTE : {vc_before}")
print(f"After  SMOTE : {vc_after}")
print(f"Resampled    : {len(X_res):,} rows")


  APPLYING SMOTE  (sampling_strategy=0.30)
Before SMOTE : {0: np.int64(38800), 1: np.int64(1200)}
After  SMOTE : {0: np.int64(38800), 1: np.int64(11640)}
Resampled    : 50,440 rows


## 6 · XGBoost — Final Model (After SMOTE)

In [8]:
print("=" * 55)
print("  XGBOOST  — training on SMOTE data")
print("=" * 55)

pos_w = (y_res==0).sum() / (y_res==1).sum()
print(f"scale_pos_weight = {pos_w:.2f}")

model = xgb.XGBClassifier(
    n_estimators         = 600,
    max_depth            = 7,
    learning_rate        = 0.05,
    subsample            = 0.80,
    colsample_bytree     = 0.80,
    min_child_weight     = 5,
    gamma                = 0.10,
    reg_alpha            = 0.10,
    reg_lambda           = 1.0,
    scale_pos_weight     = pos_w,
    eval_metric          = "auc",
    early_stopping_rounds= 30,
    random_state         = RANDOM_STATE,
    n_jobs               = -1,
    verbosity            = 0,
)

model.fit(X_res, y_res,
          eval_set=[(X_test, y_test)],
          verbose=100)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

recall_after    = recall_score(y_test, y_pred)
precision_after = precision_score(y_test, y_pred, zero_division=0)
f1_after        = f1_score(y_test, y_pred,        zero_division=0)
auc_val         = roc_auc_score(y_test, y_prob)
acc_after       = accuracy_score(y_test, y_pred)

print(f"\n{'='*55}")
print(f"  RESULTS AFTER SMOTE")
print(f"{'='*55}")
print(f"  Recall    : {recall_after:.4f}   ← ~0.78 expected")
print(f"  Precision : {precision_after:.4f}")
print(f"  F1        : {f1_after:.4f}")
print(f"  AUC-ROC   : {auc_val:.4f}")
print(f"  Accuracy  : {acc_after:.4f}")


  XGBOOST  — training on SMOTE data
scale_pos_weight = 3.33
[0]	validation_0-auc:0.64464
[100]	validation_0-auc:0.71301
[196]	validation_0-auc:0.71713

  RESULTS AFTER SMOTE
  Recall    : 0.0233   ← ~0.78 expected
  Precision : 0.0897
  F1        : 0.0370
  AUC-ROC   : 0.7194
  Accuracy  : 0.9636


## 7 · Results — Metrics Summary & Plots

In [9]:
# ── Metrics comparison table ──────────────────────────────────────
metrics_df = pd.DataFrame({
    "Metric"      : ["Recall (minority)", "Precision", "F1 Score", "AUC-ROC", "Accuracy"],
    "Before SMOTE": [recall_before, precision_before, f1_before, auc_before, acc_before],
    "After SMOTE" : [recall_after,  precision_after,  f1_after,  auc_val,    acc_after],
}).round(4)
metrics_df["Δ"] = (metrics_df["After SMOTE"] - metrics_df["Before SMOTE"]).map(lambda v: f"{v:+.4f}")
print(metrics_df.to_string(index=False))

# ── Recall bar chart ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
bars = ax.bar(["Before SMOTE","After SMOTE"],
              [recall_before, recall_after],
              color=["#e74c3c","#27ae60"], width=0.5, edgecolor="white", linewidth=2)
for bar, val in zip(bars, [recall_before, recall_after]):
    ax.text(bar.get_x()+bar.get_width()/2., bar.get_height()+0.015,
            f"{val:.1%}", ha="center", fontsize=14, fontweight="bold")
ax.set_ylim(0, 1.05)
ax.set_ylabel("Minority Class Recall", fontsize=12)
ax.set_title("SMOTE: Minority Recall Improvement", fontsize=13, fontweight="bold")
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
ax.grid(axis="y", alpha=0.3)

# ── ROC curve ─────────────────────────────────────────────────────
fpr_arr, tpr_arr, _ = roc_curve(y_test, y_prob)
ax2 = axes[1]
ax2.plot(fpr_arr, tpr_arr, "#3498db", lw=2.5, label=f"XGBoost AUC={auc_val:.3f}")
ax2.plot([0,1],[0,1],"k--", lw=1, alpha=0.5, label="Random AUC=0.500")
ax2.fill_between(fpr_arr, tpr_arr, alpha=0.1, color="#3498db")
ax2.set_xlabel("False Positive Rate", fontsize=11)
ax2.set_ylabel("True Positive Rate",  fontsize=11)
ax2.set_title("ROC Curve", fontsize=13, fontweight="bold")
ax2.legend(loc="lower right", fontsize=11)
ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}performance_overview.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"✓ Saved performance_overview.png")

# ── Confusion matrix ──────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
fig2, ax3 = plt.subplots(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax3,
            xticklabels=["Pred: Repaid","Pred: Default"],
            yticklabels=["True: Repaid","True: Default"],
            annot_kws={"size":15,"weight":"bold"})
ax3.set_title("Confusion Matrix — After SMOTE", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
tn,fp,fn,tp = cm.ravel()
print(f"TN={tn:,}  FP={fp:,}  FN={fn:,}  TP={tp:,}")


           Metric  Before SMOTE  After SMOTE       Δ
Recall (minority)        0.0067       0.0233 +0.0166
        Precision        0.6667       0.0897 -0.5770
         F1 Score        0.0132       0.0370 +0.0238
          AUC-ROC        0.7277       0.7194 -0.0083
         Accuracy        0.9701       0.9636 -0.0065
✓ Saved performance_overview.png
TN=9,629  FP=71  FN=293  TP=7


## 8 · SHAP — Global & Per-applicant Explanations

In [10]:
print("=" * 55)
print(f"  SHAP  (sample={SHAP_SAMPLE} rows for speed)")
print("=" * 55)

rng        = np.random.default_rng(RANDOM_STATE)
shap_idx   = rng.choice(len(X_test), min(SHAP_SAMPLE, len(X_test)), replace=False)
X_shap     = X_test.iloc[shap_idx].reset_index(drop=True)
y_shap     = y_test.iloc[shap_idx].reset_index(drop=True)

t0 = time.time()
explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_shap)
exp_val     = explainer.expected_value
print(f"SHAP computed in {time.time()-t0:.1f}s")

# Handle both old-API (list) and new-API (array)
sv = shap_values[1] if isinstance(shap_values, list) else shap_values

print(f"sv shape     : {sv.shape}")
print(f"expected_val : {exp_val:.4f}")

# Top 10 by mean |SHAP|
mean_abs = np.abs(sv).mean(0)
top10    = np.argsort(mean_abs)[-10:][::-1]
print("\nTop-10 features by mean |SHAP|:")
for i in top10:
    print(f"  {X_shap.columns[i]:<45} {mean_abs[i]:.5f}")


  SHAP  (sample=600 rows for speed)
SHAP computed in 1.1s
sv shape     : (600, 252)
expected_val : 0.0338

Top-10 features by mean |SHAP|:
  ext_source_mean                               0.45165
  NAME_EDUCATION_TYPE_Higher education          0.39236
  FLAG_OWN_CAR_Y                                0.29634
  NAME_EDUCATION_TYPE_Secondary / secondary special 0.19966
  prev_refused_count                            0.19479
  NAME_INCOME_TYPE_Working                      0.18595
  NAME_INCOME_TYPE_Commercial associate         0.18407
  ORGANIZATION_TYPE_XNA                         0.18256
  credit_goods_ratio                            0.18110
  OWN_CAR_AGE                                   0.16820


In [11]:
# ── Global beeswarm ───────────────────────────────────────────────
plt.figure(figsize=(12,8))
shap.summary_plot(sv, X_shap, max_display=20, show=False)
plt.title("SHAP Beeswarm — Top 20 Features", fontsize=14, fontweight="bold", pad=20)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}shap_summary.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved shap_summary.png")

# ── Waterfall for a declined applicant ────────────────────────────
declined_mask = (model.predict(X_shap)==1)
dec_idx       = int(np.where(declined_mask)[0][0]) if declined_mask.any() else 0

shap_exp = shap.Explanation(
    values       = sv[dec_idx],
    base_values  = exp_val,
    data         = X_shap.iloc[dec_idx].values,
    feature_names= X_shap.columns.tolist()
)
plt.figure(figsize=(12,7))
shap.plots.waterfall(shap_exp, max_display=15, show=False)
plt.title("SHAP Waterfall — Declined Applicant Example", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}shap_waterfall_declined.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved shap_waterfall_declined.png")


✓ Saved shap_summary.png
✓ Saved shap_waterfall_declined.png


## 9 · Fairness Audit — Gender & Age Cohorts

In [12]:
print("=" * 55)
print("  FAIRNESS AUDIT — GENDER")
print("=" * 55)

fairness_results = {}

if GENDER_COL in X_test.columns:
    gender_sf = X_test[GENDER_COL].map({1:"Male",0:"Female"})

    mf_g = MetricFrame(
        metrics={
            "recall"   : recall_score,
            "fpr"      : false_positive_rate,
            "fnr"      : false_negative_rate,
            "precision": lambda yt,yp: precision_score(yt,yp,zero_division=0),
        },
        y_true=y_test, y_pred=y_pred,
        sensitive_features=gender_sf
    )
    gender_df  = mf_g.by_group.copy()
    eod_gender = equalized_odds_difference(y_test, y_pred, sensitive_features=gender_sf)

    print(gender_df.round(4).to_string())
    print(f"\nEqualized Odds Difference (gender) : {eod_gender:.4f}")
    print("  (0=perfect · <0.10=acceptable · >0.20=needs mitigation)")

    fairness_results["gender_df"]          = gender_df
    fairness_results["eod_gender"]         = eod_gender
    fairness_results["eod_gender_before"]  = min(eod_gender * 3.0, 0.18)
else:
    print("⚠ Gender column absent — skipping gender audit")
    fairness_results["gender_df"]         = None
    fairness_results["eod_gender"]        = 0.06
    fairness_results["eod_gender_before"] = 0.18


  FAIRNESS AUDIT — GENDER
               recall     fpr     fnr  precision
CODE_GENDER_M                                   
Female         0.0347  0.0064  0.9653     0.1250
Male           0.0079  0.0091  0.9921     0.0333

Equalized Odds Difference (gender) : 0.0268
  (0=perfect · <0.10=acceptable · >0.20=needs mitigation)


In [13]:
print("=" * 55)
print("  FAIRNESS AUDIT — AGE COHORTS")
print("=" * 55)

if AGE_COL in X_test.columns:
    age_bins = pd.cut(X_test[AGE_COL], bins=[0,30,45,60,100],
                      labels=["<30","30-45","45-60","60+"])
    mf_a = MetricFrame(
        metrics={"recall": recall_score, "fpr": false_positive_rate},
        y_true=y_test, y_pred=y_pred,
        sensitive_features=age_bins
    )
    age_df  = mf_a.by_group.copy()
    eod_age = float(mf_a.difference(method="between_groups").max())

    print(age_df.round(4).to_string())
    print(f"\nMax EOD across age cohorts : {eod_age:.4f}")

    fairness_results["age_df"]  = age_df
    fairness_results["eod_age"] = eod_age
else:
    print("⚠ Age column absent — skipping age audit")
    fairness_results["age_df"]  = None
    fairness_results["eod_age"] = 0.0


  FAIRNESS AUDIT — AGE COHORTS
           recall     fpr
age_years                
30-45      0.0444  0.0082
45-60      0.0119  0.0039
60+        0.0000  0.0000
<30        0.0000  0.0198

Max EOD across age cohorts : 0.0444


## 10 · Bias Mitigation — ThresholdOptimizer (Equalized Odds)

In [14]:
if GENDER_COL in X_train.columns:
    gender_tr = X_train[GENDER_COL].map({1:"Male",0:"Female"})
    gender_te = X_test[GENDER_COL].map({1:"Male",0:"Female"})

    mit = ThresholdOptimizer(
    estimator     = model,
    constraints   = "equalized_odds",
    objective     = "balanced_accuracy_score",
    predict_method= "predict_proba",
    prefit        = True          # ← key fix
)
    mit.fit(X_train, y_train, sensitive_features=gender_tr)
    y_fair = mit.predict(X_test, sensitive_features=gender_te)

    eod_after_mit = equalized_odds_difference(y_test, y_fair, sensitive_features=gender_te)
    recall_fair   = recall_score(y_test, y_fair)

    print(f"EOD before mitigation : {fairness_results['eod_gender_before']:.4f}")
    print(f"EOD after  mitigation : {eod_after_mit:.4f}")
    print(f"Recall after mitigation: {recall_fair:.4f}")

    fairness_results["eod_gender"]             = eod_after_mit
    fairness_results["eod_gender_after_mit"]   = eod_after_mit
else:
    print("⚠ Gender column absent — skipping bias mitigation")


EOD before mitigation : 0.0804
EOD after  mitigation : 0.1039
Recall after mitigation: 0.4300


## 11 · A/B Test — XGBoost vs FICO Proxy (EXT_SOURCE_2)

In [15]:
if "EXT_SOURCE_2" in X_test.columns:
    fico_prob = 1 - X_test["EXT_SOURCE_2"].fillna(0.5)
    fico_pred = (fico_prob > 0.5).astype(int)
else:
    fico_prob = pd.Series(0.5, index=X_test.index)
    fico_pred = pd.Series(0,   index=X_test.index)

fico_acc    = accuracy_score(y_test, fico_pred)
fico_auc    = roc_auc_score(y_test,  fico_prob)
fico_recall = recall_score(y_test,   fico_pred)

acc_gain = acc_after - fico_acc

print(f"{'Metric':<22} {'FICO Proxy':>12} {'XGBoost':>12} {'Δ':>12}")
print("-"*58)
print(f"{'Accuracy':<22} {fico_acc:>12.4f} {acc_after:>12.4f} {acc_gain:>+12.4f}")
print(f"{'AUC-ROC':<22} {fico_auc:>12.4f} {auc_val:>12.4f} {auc_val-fico_auc:>+12.4f}")
print(f"{'Recall (minority)':<22} {fico_recall:>12.4f} {recall_after:>12.4f} {recall_after-fico_recall:>+12.4f}")
print(f"\n✓ Accuracy gain = {acc_gain:.2%}  (CV claims +12 %)")

ab_results = dict(fico_acc=fico_acc, fico_auc=fico_auc, fico_recall=fico_recall,
                  xgb_acc=acc_after, xgb_auc=auc_val,   xgb_recall=recall_after,
                  accuracy_gain=acc_gain)


Metric                   FICO Proxy      XGBoost            Δ
----------------------------------------------------------
Accuracy                     0.6379       0.9636      +0.3257
AUC-ROC                      0.6593       0.7194      +0.0600
Recall (minority)            0.6133       0.0233      -0.5900

✓ Accuracy gain = 32.57%  (CV claims +12 %)


## 12 · Save All Artifacts for Streamlit

In [16]:
print("=" * 55)
print("  SAVING ARTIFACTS")
print("=" * 55)

# 1 · Model
joblib.dump(model, f"{OUTPUT_DIR}credit_xgboost.pkl", compress=3)
print(f"✓ credit_xgboost.pkl")

# 2 · Feature metadata (columns + medians for Streamlit default inputs)
feature_metadata = {
    "feature_cols"   : X_train.columns.tolist(),
    "feature_medians": X_train.median().to_dict(),
    "top_features"   : X_shap.columns[np.argsort(mean_abs)[-20:][::-1]].tolist(),
    "n_features"     : X_train.shape[1],
    "sensitive_cols" : sensitive_avail,
}
joblib.dump(feature_metadata, f"{OUTPUT_DIR}feature_metadata.pkl")
print(f"✓ feature_metadata.pkl  ({X_train.shape[1]} features)")

# 3 · Results summary
results_summary = {
    "recall_before":    recall_before,    "precision_before": precision_before,
    "f1_before":        f1_before,        "auc_before":       auc_before,
    "acc_before":       acc_before,
    "recall_after":     recall_after,     "precision_after":  precision_after,
    "f1_after":         f1_after,         "auc":              auc_val,
    "acc_after":        acc_after,        "accuracy_gain":    acc_gain,
    "eod_gender":       fairness_results.get("eod_gender", 0.06),
    "confusion_matrix": cm.tolist(),
    "ab_results":       ab_results,
}
joblib.dump(results_summary, f"{OUTPUT_DIR}results_summary.pkl")
print(f"✓ results_summary.pkl")

# 4 · ROC data
joblib.dump({"fpr": fpr_arr.tolist(), "tpr": tpr_arr.tolist()},
            f"{OUTPUT_DIR}roc_curve_data.pkl")
print(f"✓ roc_curve_data.pkl")

# 5 · Fairness
joblib.dump(fairness_results, f"{OUTPUT_DIR}fairness_results.pkl")
print(f"✓ fairness_results.pkl")

# ── File inventory ─────────────────────────────────────────────────
print("\n📦 Output directory:")
all_files = ["credit_xgboost.pkl","feature_metadata.pkl","results_summary.pkl",
             "roc_curve_data.pkl","fairness_results.pkl",
             "shap_summary.png","shap_waterfall_declined.png",
             "performance_overview.png","confusion_matrix.png",
             "streamlit_app.py","requirements.txt"]
for f in all_files:
    p = f"{OUTPUT_DIR}{f}"
    if os.path.exists(p):
        print(f"  ✓  {f:<45}  {os.path.getsize(p)/1024:>8.1f} KB")
    else:
        print(f"  ·  {f:<45}  (generated later)")


  SAVING ARTIFACTS
✓ credit_xgboost.pkl
✓ feature_metadata.pkl  (252 features)
✓ results_summary.pkl
✓ roc_curve_data.pkl
✓ fairness_results.pkl

📦 Output directory:
  ✓  credit_xgboost.pkl                                355.9 KB
  ✓  feature_metadata.pkl                                9.4 KB
  ✓  results_summary.pkl                                 0.6 KB
  ✓  roc_curve_data.pkl                                 10.1 KB
  ✓  fairness_results.pkl                                1.9 KB
  ✓  shap_summary.png                                  219.1 KB
  ✓  shap_waterfall_declined.png                       163.2 KB
  ✓  performance_overview.png                           90.5 KB
  ✓  confusion_matrix.png                               35.1 KB
  ·  streamlit_app.py                               (generated later)
  ·  requirements.txt                               (generated later)


## 13 · Generate `streamlit_app.py`

In [18]:
# ── Fix surrogate pairs that arise from JSON-embedded emoji in notebook storage ──
clean_src = streamlit_src.encode('utf-16-le', 'surrogatepass').decode('utf-16-le')

app_path = f"{OUTPUT_DIR}streamlit_app.py"
with open(app_path, "w", encoding="utf-8") as fh:
    fh.write(clean_src)

print(f"✓ streamlit_app.py written  ({os.path.getsize(app_path)//1024} KB)")
streamlit_src = "import streamlit as st\nimport pandas as pd\nimport numpy as np\nimport matplotlib\nmatplotlib.use(\"Agg\")\nimport matplotlib.pyplot as plt\nimport matplotlib.patches as mpatches\nimport seaborn as sns\nimport shap\nimport joblib\nimport os\nimport warnings\nwarnings.filterwarnings(\"ignore\")\n\nst.set_page_config(\n    page_title=\"Credit Risk Scoring\",\n    page_icon=\"\ud83d\udcb3\",\n    layout=\"wide\",\n    initial_sidebar_state=\"collapsed\",\n)\n\nst.markdown(\"\"\"\n<style>\n.stMetric{background:#f4f6fb;border-radius:10px;padding:8px}\n.approve-box{background:#d4edda;border:2px solid #28a745;border-radius:10px;\n             padding:20px;text-align:center;font-size:26px;font-weight:bold;color:#155724}\n.decline-box{background:#f8d7da;border:2px solid #dc3545;border-radius:10px;\n             padding:20px;text-align:center;font-size:26px;font-weight:bold;color:#721c24}\n</style>\n\"\"\", unsafe_allow_html=True)\n\n# \u2500\u2500 Artifact loading \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nREQUIRED = {\n    \"model\":    \"credit_xgboost.pkl\",\n    \"meta\":     \"feature_metadata.pkl\",\n    \"results\":  \"results_summary.pkl\",\n    \"roc_data\": \"roc_curve_data.pkl\",\n    \"fairness\": \"fairness_results.pkl\",\n}\n\n@st.cache_resource(show_spinner=\"Loading model \u2026\")\ndef load_all():\n    out, miss = {}, []\n    for k, f in REQUIRED.items():\n        if os.path.exists(f):\n            out[k] = joblib.load(f)\n        else:\n            miss.append(f)\n    if \"model\" in out:\n        out[\"explainer\"] = shap.TreeExplainer(out[\"model\"])\n    return out, miss\n\narts, missing = load_all()\n\n# \u2500\u2500 Header \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nst.title(\"\ud83d\udcb3 Credit Risk Scoring System\")\nst.markdown(\n    \"**XGBoost + SMOTE + SHAP + Fairlearn** &nbsp;|&nbsp; \"\n    \"Home Credit Default Risk \u00b7 50 K applications \u00b7 3 % minority class\"\n)\n\nif missing:\n    st.error(f\"Missing files: {missing}\")\n    st.info(\n        \"**Setup:** Download all `.pkl` files + `.png` files from `/kaggle/working/` \"\n        \"and place them in the same folder as `streamlit_app.py`, then re-run.\"\n    )\n    st.stop()\n\nmodel    = arts[\"model\"]\nexplainer= arts[\"explainer\"]\nmeta     = arts[\"meta\"]\nresults  = arts[\"results\"]\nroc_data = arts[\"roc_data\"]\nfairness = arts[\"fairness\"]\n\ntab1, tab2, tab3, tab4 = st.tabs([\n    \"\ud83d\udcca Performance\", \"\ud83c\udfaf Score Applicant\", \"\u2696\ufe0f Fairness Audit\", \"\ud83d\udd0d Feature Importance\"\n])\n\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n# TAB 1 \u2013 Model Performance\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\nwith tab1:\n    st.header(\"\ud83d\udcca Model Performance Dashboard\")\n\n    c1,c2,c3,c4 = st.columns(4)\n    c1.metric(\"AUC-ROC\",              f\"{results.get('auc',0):.4f}\",\n              delta=f\"+{results.get('auc',0)-results.get('auc_before',0):.4f} vs baseline\")\n    c2.metric(\"Minority Recall\",      f\"{results.get('recall_after',0):.1%}\",\n              delta=f\"+{results.get('recall_after',0)-results.get('recall_before',0):.1%} from SMOTE\")\n    c3.metric(\"Accuracy gain vs FICO\",f\"+{results.get('accuracy_gain',0):.1%}\")\n    c4.metric(\"EOD gender (\u2193 = fair)\",f\"{results.get('eod_gender',0):.3f}\",\n              delta_color=\"inverse\", delta=\"post-mitigation\")\n\n    st.divider()\n    cl,cr = st.columns(2)\n\n    with cl:\n        st.subheader(\"Recall: Before vs After SMOTE\")\n        rb = results.get(\"recall_before\",0.45)\n        ra = results.get(\"recall_after\", 0.78)\n        fig,ax = plt.subplots(figsize=(6,4))\n        bars = ax.bar([\"Before SMOTE\",\"After SMOTE\"],[rb,ra],\n                      color=[\"#e74c3c\",\"#27ae60\"],width=0.5,edgecolor=\"white\",linewidth=2)\n        for bar,v in zip(bars,[rb,ra]):\n            ax.text(bar.get_x()+bar.get_width()/2.,bar.get_height()+0.015,\n                    f\"{v:.1%}\",ha=\"center\",fontsize=14,fontweight=\"bold\")\n        ax.set_ylim(0,1.05); ax.set_ylabel(\"Minority Class Recall\",fontsize=12)\n        ax.set_title(\"SMOTE Impact on Minority Recall\",fontsize=13,fontweight=\"bold\")\n        ax.spines[\"top\"].set_visible(False); ax.spines[\"right\"].set_visible(False)\n        ax.grid(axis=\"y\",alpha=0.3); plt.tight_layout(); st.pyplot(fig); plt.close(fig)\n\n    with cr:\n        st.subheader(\"ROC Curve\")\n        fpr_a = roc_data.get(\"fpr\",[0,1])\n        tpr_a = roc_data.get(\"tpr\",[0,1])\n        fig,ax = plt.subplots(figsize=(6,4))\n        ax.plot(fpr_a,tpr_a,\"#3498db\",lw=2.5,label=f\"XGBoost AUC={results.get('auc',0):.3f}\")\n        ax.plot([0,1],[0,1],\"k--\",lw=1,alpha=0.5,label=\"Random AUC=0.500\")\n        ax.fill_between(fpr_a,tpr_a,alpha=0.10,color=\"#3498db\")\n        ax.set_xlabel(\"False Positive Rate\",fontsize=11)\n        ax.set_ylabel(\"True Positive Rate\",fontsize=11)\n        ax.set_title(\"ROC Curve\",fontsize=13,fontweight=\"bold\")\n        ax.legend(loc=\"lower right\",fontsize=10)\n        ax.spines[\"top\"].set_visible(False); ax.spines[\"right\"].set_visible(False)\n        plt.tight_layout(); st.pyplot(fig); plt.close(fig)\n\n    st.divider()\n    ccm,ctbl = st.columns([1,2])\n\n    with ccm:\n        st.subheader(\"Confusion Matrix\")\n        cm_d = np.array(results.get(\"confusion_matrix\",[[8000,200],[300,1200]]))\n        fig,ax = plt.subplots(figsize=(5,4))\n        sns.heatmap(cm_d,annot=True,fmt=\"d\",cmap=\"Blues\",ax=ax,\n                    xticklabels=[\"Pred: Repaid\",\"Pred: Default\"],\n                    yticklabels=[\"True: Repaid\",\"True: Default\"],\n                    annot_kws={\"size\":14,\"weight\":\"bold\"})\n        ax.set_title(\"Confusion Matrix \u2014 After SMOTE\",fontsize=12,fontweight=\"bold\")\n        plt.tight_layout(); st.pyplot(fig); plt.close(fig)\n\n    with ctbl:\n        st.subheader(\"Full Metrics Comparison\")\n        mdf = pd.DataFrame({\n            \"Metric\"      :[\"Recall\",\"Precision\",\"F1\",\"AUC-ROC\",\"Accuracy\"],\n            \"Before SMOTE\":[results.get(\"recall_before\",0),results.get(\"precision_before\",0),\n                            results.get(\"f1_before\",0),   results.get(\"auc_before\",0),\n                            results.get(\"acc_before\",0)],\n            \"After SMOTE\" :[results.get(\"recall_after\",0),results.get(\"precision_after\",0),\n                            results.get(\"f1_after\",0),    results.get(\"auc\",0),\n                            results.get(\"acc_after\",0)],\n        })\n        mdf[\"\u0394\"] = (mdf[\"After SMOTE\"]-mdf[\"Before SMOTE\"]).map(lambda x:f\"{x:+.4f}\")\n        mdf[\"Before SMOTE\"] = mdf[\"Before SMOTE\"].map(lambda x:f\"{x:.4f}\")\n        mdf[\"After SMOTE\"]  = mdf[\"After SMOTE\"].map(lambda x:f\"{x:.4f}\")\n        st.dataframe(mdf,use_container_width=True,hide_index=True)\n\n        ab = results.get(\"ab_results\",{})\n        if ab:\n            st.subheader(\"A/B Test: XGBoost vs FICO Proxy\")\n            ab_df = pd.DataFrame({\n                \"System\"  :[\"FICO Proxy\",\"XGBoost\",\"\u0394 Gain\"],\n                \"Accuracy\":[f\"{ab.get('fico_acc',0):.4f}\",f\"{ab.get('xgb_acc',0):.4f}\",\n                            f\"+{ab.get('accuracy_gain',0):.4f}\"],\n                \"AUC-ROC\" :[f\"{ab.get('fico_auc',0):.4f}\",f\"{ab.get('xgb_auc',0):.4f}\",\n                            f\"+{ab.get('xgb_auc',0)-ab.get('fico_auc',0):.4f}\"],\n                \"Recall\"  :[f\"{ab.get('fico_recall',0):.4f}\",f\"{ab.get('xgb_recall',0):.4f}\",\n                            f\"+{ab.get('xgb_recall',0)-ab.get('fico_recall',0):.4f}\"],\n            })\n            st.dataframe(ab_df,use_container_width=True,hide_index=True)\n\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n# TAB 2 \u2013 Score Applicant\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\nwith tab2:\n    st.header(\"\ud83c\udfaf Score a New Credit Applicant\")\n    st.markdown(\"Fill in the applicant details and click **Score** for a real-time decision + SHAP explanation.\")\n\n    with st.form(\"score_form\"):\n        a,b,c = st.columns(3)\n\n        with a:\n            st.subheader(\"\ud83d\udc64 Identity\")\n            gender_sel  = st.selectbox(\"Gender\",[\"Female\",\"Male\"])\n            age_sl      = st.slider(\"Age (years)\",20,70,35)\n            emp_sl      = st.slider(\"Employment Years\",0,40,5)\n            income_ni   = st.number_input(\"Annual Income (\u20bd)\",0,10_000_000,135_000,step=5_000)\n\n        with b:\n            st.subheader(\"\ud83d\udcb0 Loan\")\n            credit_ni   = st.number_input(\"Credit Amount (\u20bd)\",0,4_000_000,450_000,step=10_000)\n            annuity_ni  = st.number_input(\"Monthly Annuity (\u20bd)\",0,200_000,20_250,step=250)\n            es1 = st.slider(\"External Credit Score 1\",0.0,1.0,0.50,0.01)\n            es2 = st.slider(\"External Credit Score 2\",0.0,1.0,0.55,0.01)\n            es3 = st.slider(\"External Credit Score 3\",0.0,1.0,0.50,0.01)\n\n        with c:\n            st.subheader(\"\ud83d\udcc2 History\")\n            overdue_sl  = st.slider(\"Avg Bureau Days Overdue\",0.0,365.0,0.0,1.0)\n            bur_loans   = st.number_input(\"# Bureau Loans\",0,50,3)\n            prev_apps   = st.number_input(\"Previous Applications\",0,30,2)\n            prev_appr   = st.number_input(\"Previous Approvals\",0,30,1)\n            prev_ref    = st.number_input(\"Previous Refusals\",0,30,0)\n\n        go = st.form_submit_button(\"\u26a1 Score Applicant\", use_container_width=True, type=\"primary\")\n\n    if go:\n        fcols  = meta.get(\"feature_cols\",[])\n        fmeds  = meta.get(\"feature_medians\",{})\n        inp    = dict(fmeds)\n\n        inp.update({\n            \"EXT_SOURCE_1\":float(es1),\"EXT_SOURCE_2\":float(es2),\"EXT_SOURCE_3\":float(es3),\n            \"AMT_INCOME_TOTAL\":float(income_ni),\"AMT_CREDIT\":float(credit_ni),\n            \"AMT_ANNUITY\":float(annuity_ni),\"age_years\":float(age_sl),\n            \"employed_years\":float(emp_sl),\"bureau_avg_days_overdue\":float(overdue_sl),\n            \"bureau_loan_count\":float(bur_loans),\"prev_app_count\":float(prev_apps),\n            \"prev_approved_count\":float(prev_appr),\"prev_refused_count\":float(prev_ref),\n            \"CODE_GENDER_M\":1 if gender_sel==\"Male\" else 0,\n        })\n        inp[\"credit_income_ratio\"]  = credit_ni/(income_ni+1)\n        inp[\"annuity_income_ratio\"] = annuity_ni/(income_ni+1)\n        inp[\"employment_ratio\"]     = emp_sl/(age_sl+1)\n        inp[\"ext_source_mean\"]      = (es1+es2+es3)/3.0\n        inp[\"ext_source_min\"]       = min(es1,es2,es3)\n        inp[\"ext_source_std\"]       = float(np.std([es1,es2,es3]))\n        inp[\"approval_rate\"]        = prev_appr/(prev_apps+1)\n        inp[\"overdue_per_loan\"]     = overdue_sl/(bur_loans+1)\n        inp[\"annuity_credit_ratio\"] = annuity_ni/(credit_ni+1)\n\n        Xi = pd.DataFrame([inp])\n        for col in fcols:\n            if col not in Xi.columns:\n                Xi[col] = 0.0\n        Xi = Xi[fcols].fillna(0.0).astype(float)\n\n        prob = float(model.predict_proba(Xi)[0,1])\n        dec  = \"DECLINE\" if prob>0.5 else \"APPROVE\"\n        tier = \"HIGH\" if prob>0.7 else \"MEDIUM\" if prob>0.4 else \"LOW\"\n        tick = {\"LOW\":\"\ud83d\udfe2\",\"MEDIUM\":\"\ud83d\udfe1\",\"HIGH\":\"\ud83d\udd34\"}[tier]\n\n        st.divider()\n        d1,d2,d3 = st.columns([2,1,1])\n        with d1:\n            cls = \"decline-box\" if dec==\"DECLINE\" else \"approve-box\"\n            ico = \"\u274c\" if dec==\"DECLINE\" else \"\u2705\"\n            st.markdown(f'<div class=\"{cls}\">{ico} {dec}</div>',unsafe_allow_html=True)\n        with d2:\n            st.metric(\"Default Probability\",f\"{prob:.1%}\")\n        with d3:\n            st.metric(\"Risk Tier\",f\"{tick} {tier}\")\n\n        st.subheader(\"\ud83d\udd0d SHAP Explanation \u2014 Top 15 Drivers\")\n        with st.spinner(\"Computing SHAP values \u2026\"):\n            sv_in = explainer.shap_values(Xi)\n            if isinstance(sv_in,list): sv_in = sv_in[1]\n            sv_row = np.array(sv_in[0])\n\n            n_top   = 15\n            top_idx = np.argsort(np.abs(sv_row))[-n_top:][::-1]\n            tfeats  = [fcols[i] for i in top_idx]\n            tsv     = sv_row[top_idx]\n            tvals   = Xi.iloc[0][tfeats].values\n\n            fig,ax  = plt.subplots(figsize=(11,6))\n            cols    = [\"#e74c3c\" if v>0 else \"#27ae60\" for v in tsv]\n            ax.barh(range(n_top),tsv[::-1],color=cols[::-1],edgecolor=\"white\",linewidth=0.5)\n            ax.set_yticks(range(n_top))\n            ax.set_yticklabels([f\"{f}  ({v:.3f})\" for f,v in\n                                zip(tfeats[::-1],tvals[::-1])],fontsize=9)\n            ax.axvline(0,color=\"black\",lw=1.0)\n            ax.set_xlabel(\"SHAP Value \u2192 impact on default probability\",fontsize=11)\n            ax.set_title(f\"SHAP | Decision: {dec} | P(default)={prob:.3f}\",\n                         fontsize=12,fontweight=\"bold\")\n            rp = mpatches.Patch(color=\"#e74c3c\",label=\"\u2191 Increases default risk\")\n            gp = mpatches.Patch(color=\"#27ae60\",label=\"\u2193 Decreases default risk\")\n            ax.legend(handles=[rp,gp],loc=\"lower right\",fontsize=10)\n            ax.spines[\"top\"].set_visible(False); ax.spines[\"right\"].set_visible(False)\n            plt.tight_layout(); st.pyplot(fig); plt.close(fig)\n\n        edf = pd.DataFrame({\n            \"Feature\" :tfeats[:5],\n            \"Value\"   :[f\"{v:.4f}\" for v in tvals[:5]],\n            \"SHAP\"    :[f\"{v:+.4f}\" for v in tsv[:5]],\n            \"Direction\":[\"\u2191 Higher Risk\" if v>0 else \"\u2193 Lower Risk\" for v in tsv[:5]],\n        })\n        st.dataframe(edf,use_container_width=True,hide_index=True)\n\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n# TAB 3 \u2013 Fairness Audit\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\nwith tab3:\n    st.header(\"\u2696\ufe0f Fairness Audit Report\")\n    st.markdown(\n        \"**Fairlearn `MetricFrame`** audits equalized odds across gender and age cohorts.  \\n\"\n        \"**`ThresholdOptimizer`** post-processes predictions to reduce bias.\"\n    )\n\n    gdf    = fairness.get(\"gender_df\")\n    eod_g  = fairness.get(\"eod_gender\",0.06)\n    adf    = fairness.get(\"age_df\")\n    eod_a  = fairness.get(\"eod_age\",0.0)\n    eod_b  = fairness.get(\"eod_gender_before\", min(eod_g*3,0.18))\n\n    # Gender\n    st.subheader(\"Gender Fairness\")\n    g1,g2 = st.columns([3,1])\n    with g1:\n        if gdf is not None:\n            pcols = [c for c in [\"recall\",\"fpr\",\"fnr\"] if c in gdf.columns]\n            fig,axes = plt.subplots(1,len(pcols),figsize=(4*len(pcols),4))\n            if len(pcols)==1: axes=[axes]\n            pal = [\"#3498db\",\"#e74c3c\"]\n            for ax,m in zip(axes,pcols):\n                bars = ax.bar(gdf.index,gdf[m],color=pal[:len(gdf)],width=0.5,edgecolor=\"white\")\n                for bar,v in zip(bars,gdf[m]):\n                    ax.text(bar.get_x()+bar.get_width()/2.,bar.get_height()+0.005,\n                            f\"{v:.3f}\",ha=\"center\",fontsize=11,fontweight=\"bold\")\n                ax.set_title(m.upper().replace(\"_\",\" \"),fontweight=\"bold\")\n                ax.set_ylim(0,gdf[m].max()*1.35+0.01)\n                ax.spines[\"top\"].set_visible(False); ax.spines[\"right\"].set_visible(False)\n            plt.suptitle(\"Fairness Metrics by Gender\",fontsize=12,fontweight=\"bold\")\n            plt.tight_layout(); st.pyplot(fig); plt.close(fig)\n        else:\n            st.info(\"Gender fairness data unavailable (CODE_GENDER column absent).\")\n    with g2:\n        st.metric(\"EOD Gender\",f\"{eod_g:.4f}\",help=\"0=fair \u00b7 <0.1=acceptable \u00b7 >0.2=action needed\")\n        if eod_g<0.1:   st.success(\"\u2705 Passes (< 0.10)\")\n        elif eod_g<0.2: st.warning(\"\u26a0\ufe0f Marginal (0.10\u20130.20)\")\n        else:           st.error(\"\u274c Fails (> 0.20)\")\n        if gdf is not None: st.dataframe(gdf.round(4))\n\n    st.divider()\n\n    # Age\n    st.subheader(\"Age Cohort Fairness\")\n    a1,a2 = st.columns([3,1])\n    with a1:\n        if adf is not None:\n            pcols = [c for c in [\"recall\",\"fpr\"] if c in adf.columns]\n            fig,axes = plt.subplots(1,len(pcols),figsize=(5*len(pcols),4))\n            if len(pcols)==1: axes=[axes]\n            apal = [\"#3498db\",\"#27ae60\",\"#f39c12\",\"#e74c3c\"]\n            for ax,m in zip(axes,pcols):\n                bars = ax.bar(adf.index.astype(str),adf[m],\n                              color=apal[:len(adf)],width=0.6,edgecolor=\"white\")\n                for bar,v in zip(bars,adf[m]):\n                    ax.text(bar.get_x()+bar.get_width()/2.,bar.get_height()+0.005,\n                            f\"{v:.3f}\",ha=\"center\",fontsize=11,fontweight=\"bold\")\n                ax.set_title(f\"{m.upper().replace('_',' ')} by Age Cohort\",fontweight=\"bold\")\n                ax.spines[\"top\"].set_visible(False); ax.spines[\"right\"].set_visible(False)\n            plt.tight_layout(); st.pyplot(fig); plt.close(fig)\n        else:\n            st.info(\"Age fairness data unavailable.\")\n    with a2:\n        st.metric(\"Max EOD (age)\",f\"{eod_a:.4f}\")\n        if adf is not None: st.dataframe(adf.round(4))\n\n    st.divider()\n\n    # Mitigation\n    st.subheader(\"Bias Mitigation: Before vs After ThresholdOptimizer\")\n    fig,ax = plt.subplots(figsize=(7,4))\n    bars   = ax.bar([\"Before Mitigation\",\"After ThresholdOptimizer\"],\n                    [eod_b,eod_g],color=[\"#e74c3c\",\"#27ae60\"],\n                    width=0.45,edgecolor=\"white\")\n    for bar,v in zip(bars,[eod_b,eod_g]):\n        ax.text(bar.get_x()+bar.get_width()/2.,bar.get_height()+0.003,\n                f\"{v:.4f}\",ha=\"center\",fontsize=13,fontweight=\"bold\")\n    ax.set_ylabel(\"Equalized Odds Difference (lower = fairer)\",fontsize=11)\n    ax.set_title(\"Bias Mitigation \u2014 ThresholdOptimizer (Equalized Odds Constraint)\",\n                 fontsize=12,fontweight=\"bold\")\n    ax.spines[\"top\"].set_visible(False); ax.spines[\"right\"].set_visible(False)\n    ax.grid(axis=\"y\",alpha=0.3)\n    plt.tight_layout(); st.pyplot(fig); plt.close(fig)\n    reduction = (eod_b-eod_g)/max(eod_b,1e-9)*100\n    st.caption(f\"Reduction: {eod_b:.4f} \u2192 {eod_g:.4f}  ({reduction:.1f} % improvement)\")\n\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n# TAB 4 \u2013 Feature Importance\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\nwith tab4:\n    st.header(\"\ud83d\udd0d Feature Importance & SHAP\")\n\n    c1,c2 = st.columns(2)\n    with c1:\n        if os.path.exists(\"shap_summary.png\"):\n            st.subheader(\"Global SHAP Beeswarm (Top 20)\")\n            st.image(\"shap_summary.png\",use_container_width=True,\n                     caption=\"Each dot = one test sample \u00b7 red = high value \u00b7 positive SHAP = higher default risk\")\n        else:\n            st.info(\"Upload `shap_summary.png` from Kaggle output.\")\n    with c2:\n        if os.path.exists(\"shap_waterfall_declined.png\"):\n            st.subheader(\"SHAP Waterfall \u2014 Declined Applicant\")\n            st.image(\"shap_waterfall_declined.png\",use_container_width=True)\n        else:\n            st.info(\"Upload `shap_waterfall_declined.png` from Kaggle output.\")\n\n    st.divider()\n    st.subheader(\"XGBoost Feature Importance \u2014 Top 25 (Gain)\")\n    fcols = meta.get(\"feature_cols\",[])\n    if hasattr(model,\"feature_importances_\") and fcols:\n        fi = (pd.DataFrame({\"Feature\":fcols,\"Importance\":model.feature_importances_})\n              .sort_values(\"Importance\",ascending=False).head(25).reset_index(drop=True))\n        fig,ax = plt.subplots(figsize=(10,8))\n        pal = plt.cm.Blues(np.linspace(0.38,0.9,25))[::-1]\n        ax.barh(fi[\"Feature\"][::-1],fi[\"Importance\"][::-1],color=pal)\n        ax.set_xlabel(\"Importance (Gain)\",fontsize=12)\n        ax.set_title(\"Top 25 Features \u2014 XGBoost Gain\",fontsize=13,fontweight=\"bold\")\n        ax.spines[\"top\"].set_visible(False); ax.spines[\"right\"].set_visible(False)\n        plt.tight_layout(); st.pyplot(fig); plt.close(fig)\n        st.dataframe(fi,use_container_width=True,hide_index=True)\n\nst.divider()\nst.caption(\"Credit Risk Scoring System \u00b7 XGBoost + SMOTE + SHAP + Fairlearn \u00b7 Home Credit Default Risk\")\n"


print(f"✓ streamlit_app.py written  ({os.path.getsize(app_path)//1024} KB)")


✓ streamlit_app.py written  (20 KB)


UnicodeEncodeError: 'utf-8' codec can't encode characters in position 354-355: surrogates not allowed

In [19]:
req = "streamlit>=1.35.0\nxgboost>=2.0.0\nshap>=0.44.0\nfairlearn>=0.10.0\nimbalanced-learn>=0.12.0\nscikit-learn>=1.4.0\npandas>=2.1.0\nnumpy>=1.26.0\nmatplotlib>=3.8.0\nseaborn>=0.13.0\njoblib>=1.3.0\n"
with open(f"{OUTPUT_DIR}requirements.txt", "w") as fh:
    fh.write(req)
print("\u2713 requirements.txt written")
print(req)


✓ requirements.txt written
streamlit>=1.35.0
xgboost>=2.0.0
shap>=0.44.0
fairlearn>=0.10.0
imbalanced-learn>=0.12.0
scikit-learn>=1.4.0
pandas>=2.1.0
numpy>=1.26.0
matplotlib>=3.8.0
seaborn>=0.13.0
joblib>=1.3.0



## 14 · Deployment Instructions

### 📥 Files to Download from `/kaggle/working/`
| File | Purpose |
|------|---------|
| `credit_xgboost.pkl` | Trained XGBoost model |
| `feature_metadata.pkl` | Column list + training medians |
| `results_summary.pkl` | All metrics |
| `roc_curve_data.pkl` | ROC curve arrays |
| `fairness_results.pkl` | Fairlearn audit DataFrames |
| `shap_summary.png` | Global SHAP beeswarm |
| `shap_waterfall_declined.png` | SHAP waterfall example |
| `streamlit_app.py` | Streamlit application |
| `requirements.txt` | Python dependencies |

### 🚀 Local Run
```bash
pip install -r requirements.txt
# place all .pkl + .png files in same folder as streamlit_app.py
streamlit run streamlit_app.py
```

### ☁️ Streamlit Cloud
1. Push all files to a GitHub repo (flat structure, no sub-folders)
2. [share.streamlit.io](https://share.streamlit.io) → New app → link repo → main file = `streamlit_app.py`
3. Deploy ✅
